# ETS Model Benchmark — smooth 1.1.1 (2026-10-01)

Rerun of the ETS comparison on M1, M3 and Tourism (5,315 series) with
**smooth 1.1.1**, against the previous smooth run of 2026-09-24 (smooth 1.0.9,
`2026-09-24-smooth-1-0-9-benchmark_exponential_smoothing.ipynb`).

**Only the smooth rows were recomputed**, and only the optimal and backcasting
initialisations: `ADAM ETS Back/Opt` and `ES Back/Opt` (`model="ZXZ"`, `ZXN` on
non-seasonal data). The competitor rows are not refitted: RMSSE and the scaled
pinball are read from `smooth-paper/Data/python-ets-rmsse.csv` and
`python-ets-pinball.csv`; their SAME is not in those files and comes from
`2026-07-20-benchmark-ets-all.npy`. The previous smooth rows come from
`2026-09-24-benchmark-ets-smooth.npy`; their pinball array is not in the
repository, so there is no pinball "before".

One fit per (method, series) gives the point forecast and the 99 quantiles
(`interval="prediction"`, `side="both"`), so RMSSE, SAME and pinball all come from
the same model.

**Timing caveat.** This run used 4 cores and Python 3.14, so times are not
comparable with the saved runs and are not discussed.

In [1]:
import os, glob, time, signal, datetime, subprocess, sys, warnings, multiprocessing
from concurrent.futures import ProcessPoolExecutor, as_completed
import numpy as np
import pandas as pd
warnings.filterwarnings("ignore")
pd.set_option("display.width", 220)

HERE = os.getcwd()
DATA = os.path.join(HERE, "smooth-paper", "Data")
LEVELS = np.round(np.arange(0.01, 1.0, 0.01), 2)               # 99 quantile levels
TWO = [round(1 - 2 * t, 2) for t in LEVELS if t < 0.5]          # two-sided interval levels
VERSION_TAG = "smooth-1-1-1"
TASK_TIMEOUT = int(os.environ.get("BENCH_TASK_TIMEOUT", "120"))
DATE = "2026-10-01"
KIND = "ets"

## Versions

In [2]:
from importlib.metadata import version as pkg_version

print(f"python {sys.version.split()[0]}   cores {multiprocessing.cpu_count()}")
for _p in ("smooth", "greybox", "numpy", "pandas", "fcompdata"):
    print(f"{_p:12s} v{pkg_version(_p)}")
print("\ngit HEAD:", subprocess.run(["git", "log", "-1", "--format=%h %s"],
                                  capture_output=True, text=True).stdout.strip())

python 3.14.4   cores 4
smooth       v1.1.1
greybox      v1.0.7
numpy        v2.5.3
pandas       v3.0.6
fcompdata    v0.1.2

git HEAD: 92f3961e Fix sign-compare warnings in adamCore::reapply loops


## Datasets

In [3]:
from fcompdata import M1, M3, Tourism

datasets = ([M1[k] for k in M1.keys()]
            + [M3[k] for k in M3.keys()]
            + [Tourism[k] for k in Tourism.keys()])
print(f"M1: {len(M1)}   M3: {len(M3)}   Tourism: {len(Tourism)}   total: {len(datasets)}")

M1: 1001   M3: 3003   Tourism: 1311   total: 5315


## Methods and harness

In [4]:
METHODS = [
    ("ADAM ETS Back", {"class": "ADAM", "initial": "backcasting"}),
    ("ADAM ETS Opt", {"class": "ADAM", "initial": "optimal"}),
    ("ES Back", {"class": "ES", "initial": "backcasting"}),
    ("ES Opt", {"class": "ES", "initial": "optimal"}),
]
SMOOTH = [name for name, _ in METHODS]


def fit_predict(cfg, s):
    from smooth import ADAM, ES
    model = "ZXZ" if s.period > 1 else "ZXN"
    m = (ADAM if cfg["class"] == "ADAM" else ES)(model=model, lags=lags_of(s),
                                                  initial=cfg["initial"])
    m.fit(np.asarray(s.x, float))
    return quantiles(m.predict(h=s.h, interval="prediction", level=TWO, side="both"))

In [5]:
def lags_of(s):
    return [1, s.period] if s.period > 1 else [1]


def quantiles(fc):
    # The 99 quantiles from the two-sided intervals, with the mean at 0.5.
    mean = np.asarray(fc.mean, float).ravel()
    lower, upper = np.asarray(fc.lower, float), np.asarray(fc.upper, float)
    lo = {round(float(c), 2): i for i, c in enumerate(fc.lower.columns)}
    up = {round(float(c), 2): i for i, c in enumerate(fc.upper.columns)}
    Q = np.empty((mean.size, 99))
    for k, t in enumerate(LEVELS):
        if abs(t - 0.5) < 1e-9:
            Q[:, k] = mean
        elif t < 0.5:
            Q[:, k] = lower[:, lo[round(t, 2)]]
        else:
            Q[:, k] = upper[:, up[round(t, 2)]]
    return mean, Q


def score(s, mean, Q):
    # RMSSE, SAME and the scaled pinball averaged over the 99 levels.
    x, y = np.asarray(s.x, float), np.asarray(s.xx, float)
    scale_sq, scale_abs = np.mean(np.diff(x) ** 2), np.mean(np.abs(np.diff(x)))
    if not np.all(np.isfinite(mean)):
        return np.nan, np.nan, np.nan
    rmsse = np.sqrt(np.mean((y - mean) ** 2) / scale_sq) if scale_sq > 0 else np.nan
    same = np.abs(np.mean(y - mean)) / scale_abs if scale_abs > 0 else np.nan
    pinball = np.nan
    if Q is not None and np.all(np.isfinite(Q)):
        diff = y[:, None] - np.sort(Q, axis=1)
        loss = np.maximum(LEVELS * diff, (LEVELS - 1) * diff).mean(axis=0)
        pinball = np.mean(loss) / (scale_abs if scale_abs > 0 else 1.0)
    return rmsse, same, pinball


def _alarm(signum, frame):
    raise TimeoutError("task timeout")


def task(args):
    # One (method, series) fit: returns rmsse, same, time, pinball.
    warnings.filterwarnings("ignore")
    j, i, s, cfg = args
    out = np.full(4, np.nan)
    try:
        signal.signal(signal.SIGALRM, _alarm)
        signal.alarm(TASK_TIMEOUT)
        start = time.time()
        mean, Q = fit_predict(cfg, s)
        out[2] = time.time() - start
        out[0], out[1], out[3] = score(s, mean[: s.h], None if Q is None else Q[: s.h])
    except Exception:
        pass
    finally:
        signal.alarm(0)
    return j, i, out


def run(methods, workers):
    result = np.full((len(methods), len(datasets), 4), np.nan)
    tasks = [(j, i, s, cfg) for j, (_, cfg) in enumerate(methods) for i, s in enumerate(datasets)]
    start = time.time()
    ctx = multiprocessing.get_context("fork")
    with ProcessPoolExecutor(max_workers=workers, mp_context=ctx) as ex:
        for done, f in enumerate(as_completed([ex.submit(task, t) for t in tasks]), 1):
            j, i, out = f.result()
            result[j, i] = out
            if done % 2000 == 0:
                print(f"  {done}/{len(tasks)}  {(time.time() - start) / 60:.1f} min", flush=True)
    print(f"done in {(time.time() - start) / 60:.1f} min")
    return result

## Run or load

In [6]:
# Fit the smooth rows on the current build, or load the saved 1.1.1 array
# (BENCH_RERUN=1 forces a refit). Columns: rmsse, same, time, pinball.
saved_file = os.path.join(HERE, f"{DATE}-benchmark-{KIND}-{VERSION_TAG}.npy")
if os.environ.get("BENCH_RERUN", "0") == "1" or not os.path.exists(saved_file):
    workers = int(os.environ.get("BENCH_WORKERS", multiprocessing.cpu_count()))
    print(f"{len(METHODS)} methods x {len(datasets)} series, {workers} workers")
    current = run(METHODS, workers)
    np.save(saved_file, current)
else:
    current = np.load(saved_file)
print(os.path.basename(saved_file), current.shape)

4 methods x 5315 series, 4 workers


  2000/21260  1.8 min


  4000/21260  3.6 min


  6000/21260  7.9 min


  8000/21260  12.4 min


...

  14000/21260  27.1 min


  16000/21260  30.6 min


  18000/21260  39.5 min


  20000/21260  48.3 min


done in 53.5 min
2026-10-01-benchmark-ets-smooth-1-1-1.npy (4, 5315, 4)


## Saved runs

In [7]:
# The current run, the previous smooth run and the competitors (saved files).
RESULT = {name: {"rmsse": current[j, :, 0], "same": current[j, :, 1],
                 "pinball": current[j, :, 3], "source": DATE}
          for j, name in enumerate(SMOOTH)}

prev_names = ["ADAM ETS Back", "ADAM ETS Opt", "ADAM ETS Two", "ES Back", "ES Opt", "ES Two",
              "ES XXX", "ES ZZZ", "ES FFF", "ES SXS"]
prev = np.load("2026-09-24-benchmark-ets-smooth.npy")
PREVIOUS_LABEL = "2026-09-24 (smooth 1.0.9)"
PREVIOUS = {name: {"rmsse": prev[prev_names.index(name), :, 0],
                   "same": prev[prev_names.index(name), :, 1], "pinball": None}
            for name in SMOOTH}

all_names = prev_names + ["statsforecast AutoETS", "sktime AutoETS", "skforecast AutoETS",
                          "aeon AutoETS"]
same_saved = np.load("2026-07-20-benchmark-ets-all.npy")
rmsse_csv = pd.read_csv(os.path.join(DATA, "python-ets-rmsse.csv"))
pinball_csv = pd.read_csv(os.path.join(DATA, "python-ets-pinball.csv"))
for name in rmsse_csv.columns:
    RESULT[name] = {"rmsse": rmsse_csv[name].to_numpy(),
                    "same": same_saved[all_names.index(name), :, 1],
                    "pinball": pinball_csv[name].to_numpy(),
                    "source": "csv (SAME: 2026-07-20)"}
for name in SMOOTH:
    RESULT[name + " (1.0.9)"] = dict(PREVIOUS[name], source="2026-09-24")

## Results summary

In [8]:
def metric_table(column, label):
    names = list(RESULT)
    values = {n: RESULT[n][column] for n in names}
    frame = pd.DataFrame({
        "Method": names,
        "Source": [RESULT[n]["source"] for n in names],
        "Min": [np.nanmin(values[n]) if values[n] is not None else np.nan for n in names],
        "Q1": [np.nanquantile(values[n], 0.25) if values[n] is not None else np.nan for n in names],
        "Med": [np.nanmedian(values[n]) if values[n] is not None else np.nan for n in names],
        "Q3": [np.nanquantile(values[n], 0.75) if values[n] is not None else np.nan for n in names],
        "Max": [np.nanmax(values[n]) if values[n] is not None else np.nan for n in names],
        "Mean": [np.nanmean(values[n]) if values[n] is not None else np.nan for n in names],
        "Failed": [int(np.sum(np.isnan(values[n]))) if values[n] is not None else np.nan
                   for n in names],
    })
    print("\n" + "=" * 110 + f"\n{label}, all {len(datasets)} series (lower = better)\n" + "=" * 110)
    print(frame.to_string(index=False, float_format=lambda x: f"{x:.4f}"))
    return frame


summary_rmsse = metric_table("rmsse", "RMSSE")


RMSSE, all 5315 series (lower = better)
               Method                 Source    Min     Q1    Med     Q3            Max        Mean  Failed
        ADAM ETS Back             2026-10-01 0.0246 0.6724 1.1867 2.3416        51.6160      1.9461       0
         ADAM ETS Opt             2026-10-01 0.0221 0.6681 1.1809 2.3386        51.6160      1.9489       0
              ES Back             2026-10-01 0.0245 0.6799 1.1881 2.3172        51.6160      1.9469       0
               ES Opt             2026-10-01 0.0221 0.6707 1.1902 2.3597        51.6160      1.9477       0
statsforecast AutoETS csv (SAME: 2026-07-20) 0.0245 0.6732 1.1892 2.3267        51.6160      1.9237       0
       sktime AutoETS csv (SAME: 2026-07-20) 0.0245 0.6760 1.1902 2.4653 565753206.7206 106608.7034       0
   skforecast AutoETS csv (SAME: 2026-07-20) 0.0747 0.7472 1.3449 2.7211        50.5434      2.2737       0
         aeon AutoETS csv (SAME: 2026-07-20) 0.0405 0.7092 1.2389 2.5010        51.6160      2.

In [9]:
summary_same = metric_table("same", "SAME")


SAME, all 5315 series (lower = better)
               Method                 Source    Min     Q1    Med     Q3            Max       Mean  Failed
        ADAM ETS Back             2026-10-01 0.0003 0.3852 1.0160 2.4249        56.2820     1.9673       0
         ADAM ETS Opt             2026-10-01 0.0001 0.3802 1.0128 2.4606        55.1018     1.9691       0
              ES Back             2026-10-01 0.0000 0.3879 1.0339 2.4228        56.5176     1.9674       0
               ES Opt             2026-10-01 0.0001 0.3752 1.0109 2.4482        54.6860     1.9677       0
statsforecast AutoETS csv (SAME: 2026-07-20) 0.0002 0.3754 1.0152 2.4350        53.6136     1.9387       0
       sktime AutoETS csv (SAME: 2026-07-20) 0.0004 0.3927 1.0251 2.5732 385286937.8236 72714.2063       0
   skforecast AutoETS csv (SAME: 2026-07-20) 0.0010 0.4571 1.2178 2.9540        59.8444     2.4099       0
         aeon AutoETS csv (SAME: 2026-07-20) 0.0000 0.3962 1.0540 2.5876        51.3012     2.0383      

In [10]:
summary_pinball = metric_table("pinball", "Scaled pinball (mean over 99 levels)")


Scaled pinball (mean over 99 levels), all 5315 series (lower = better)
               Method                 Source    Min     Q1    Med     Q3                           Max                      Mean    Failed
        ADAM ETS Back             2026-10-01 0.0149 0.2887 0.5048 0.9612                       25.4511                    0.8592    0.0000
         ADAM ETS Opt             2026-10-01 0.0147 0.2881 0.5042 0.9747                       24.7495                    0.8569    0.0000
              ES Back             2026-10-01 0.0147 0.2936 0.5133 0.9673                       25.3333                    0.8497    0.0000
               ES Opt             2026-10-01 0.0152 0.2903 0.5096 0.9867                       24.2236                    0.8483    0.0000
statsforecast AutoETS csv (SAME: 2026-07-20) 0.0150 0.2930 0.5115 0.9711                       22.8934                    0.8495    0.0000
       sktime AutoETS csv (SAME: 2026-07-20) 0.0139 0.2883 0.5091 1.0290                118996

## Change against the previous smooth run

The same series, before (smooth 1.0.9) and after (smooth 1.1.1). `Better`/`Worse` count the series where the metric went down/up.

In [11]:
def compare(column):
    rows = []
    for name in SMOOTH:
        before, after = PREVIOUS[name][column], RESULT[name][column]
        row = {"Method": name, "Metric": column.upper()}
        for tag, v in (("before", before), ("after", after)):
            row[f"Mean {tag}"] = np.nanmean(v) if v is not None else np.nan
            row[f"Med {tag}"] = np.nanmedian(v) if v is not None else np.nan
        if before is not None:
            d = after - before
            row["Better"], row["Worse"] = int(np.sum(d < -1e-8)), int(np.sum(d > 1e-8))
        rows.append(row)
    return rows


rows = [r for column in ("rmsse", "same", "pinball") for r in compare(column)]
print(f"before = {PREVIOUS_LABEL}   after = {DATE} (smooth {pkg_version('smooth')})")
change = pd.DataFrame(rows)
print(change.to_string(index=False, float_format=lambda x: f"{x:.4f}"))

before = 2026-09-24 (smooth 1.0.9)   after = 2026-10-01 (smooth 1.1.1)
       Method  Metric  Mean before  Med before  Mean after  Med after   Better    Worse
ADAM ETS Back   RMSSE       1.9472      1.1868      1.9461     1.1867 697.0000 762.0000
 ADAM ETS Opt   RMSSE       1.9489      1.1809      1.9489     1.1809   8.0000  14.0000
      ES Back   RMSSE       1.9441      1.1893      1.9469     1.1881 723.0000 772.0000
       ES Opt   RMSSE       1.9477      1.1902      1.9477     1.1902   8.0000  14.0000
ADAM ETS Back    SAME       1.9720      1.0141      1.9673     1.0160 718.0000 751.0000
 ADAM ETS Opt    SAME       1.9691      1.0123      1.9691     1.0128   9.0000  13.0000
      ES Back    SAME       1.9658      1.0310      1.9674     1.0339 731.0000 774.0000
       ES Opt    SAME       1.9676      1.0103      1.9677     1.0109   9.0000  13.0000
ADAM ETS Back PINBALL          NaN         NaN      0.8592     0.5048      NaN      NaN
 ADAM ETS Opt PINBALL          NaN         NaN   